# Build the Dataset A canonical node map on Kaggle

This Kaggle notebook independently derives the sole authoritative node universe from Dataset A. It never searches for, reads, compares, converts, renames, overwrites, or deletes any legacy node-map file. It never reads Dataset B.

The workflow is sequential and fail-closed. Run each cell in order. Google Drive is accessed through the Drive API because Kaggle cannot mount Drive. Completed-workbook checkpoints are written to the private Drive output folder and are reused only after repository, parser, inventory, size, and checksum validation.

Sections are explicit: runtime setup (Stages 1, 3, and 4), secure authentication and storage (Stage 2), Drive data access (Stage 5), checkpoint restore (Stage 6), processing (Stages 7 through 10), and persistent output publication and verification (Stages 11 through 16).

## Stage 1 - Configure the Kaggle runtime

Enable Internet access in the Kaggle notebook settings before running this stage. Kaggle runtime files under `/kaggle/working` are ephemeral; authoritative checkpoints and outputs are uploaded to Google Drive.

This replaces Colab-only mounting with Kaggle-compatible packages. It does not authenticate or read Drive yet.

**Stop conditions:** stop outside Kaggle, when Internet access is disabled, or if dependency installation fails.

In [ ]:
import subprocess
import sys
from pathlib import Path

KAGGLE_WORKING_ROOT = Path("/kaggle/working")
assert KAGGLE_WORKING_ROOT.is_dir(), (
    "STOP: This notebook requires a Kaggle Notebook runtime."
)
# Install into the active Kaggle kernel instead of an ambiguous shell pip.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "google-api-python-client>=2.100",
        "google-auth>=2.30",
        "google-auth-httplib2>=0.2",
    ],
    check=True,
)
print("Kaggle working root:", KAGGLE_WORKING_ROOT)
print("Python executable:", sys.executable)
print("PASS: Kaggle runtime dependencies are available.")

## Stage 2 - Authenticate securely and define storage locations

Kaggle cannot mount Google Drive. This notebook uses the Google Drive API. Create these Kaggle Secrets before running:

- `GOOGLE_DRIVE_CREDENTIALS_JSON`: complete Google `authorized_user` JSON for ordinary My Drive (recommended), or service-account JSON when the output is on a writable Shared Drive;
- `GOOGLE_DRIVE_DATASET_A_FOLDER_ID`: the shared Dataset A folder ID;
- `GOOGLE_DRIVE_OUTPUT_FOLDER_ID`: the shared private `TDMEC_PROJECT_OUTPUTS` folder ID.

`gdown` is not used: the authenticated Drive API supports private folders, chunked retries, checksum metadata, checkpoint uploads, and persistent output writes in one controlled interface.

Generate the `authorized_user` credential on a trusted local machine with a Google OAuth Desktop client and the full Drive scope, then store the resulting JSON only as a private Kaggle Secret. Do not upload the credential JSON as a Kaggle Dataset or notebook file.

An `authorized_user` credential uses the Drive owner's quota and is the reliable choice for an output folder in ordinary My Drive. A service account has no personal Drive storage quota, so use that credential type only when the output folder is on a Shared Drive that permits it to add files. Credentials are read from Kaggle Secrets, retained only in memory, never printed, and never written to disk.

**Stop conditions:** stop on a missing secret, malformed credential, inaccessible folder, or unexpected folder type.

In [ ]:
import hashlib
import io
import json

from google.oauth2 import service_account
from google.oauth2.credentials import Credentials as UserCredentials
from googleapiclient.discovery import build
from kaggle_secrets import UserSecretsClient

SECRET_CREDENTIALS = "GOOGLE_DRIVE_CREDENTIALS_JSON"
SECRET_DATASET_A_FOLDER = "GOOGLE_DRIVE_DATASET_A_FOLDER_ID"
SECRET_OUTPUT_FOLDER = "GOOGLE_DRIVE_OUTPUT_FOLDER_ID"
DRIVE_SCOPE = "https://www.googleapis.com/auth/drive"

# Kaggle Secrets keeps OAuth material out of notebook source and outputs.
secret_client = UserSecretsClient()
credentials_text = secret_client.get_secret(SECRET_CREDENTIALS)
DATASET_A_FOLDER_ID = secret_client.get_secret(SECRET_DATASET_A_FOLDER)
OUTPUT_FOLDER_ID = secret_client.get_secret(SECRET_OUTPUT_FOLDER)
assert credentials_text, "STOP: Google Drive credential secret is missing."
assert DATASET_A_FOLDER_ID, "STOP: Dataset A folder-ID secret is missing."
assert OUTPUT_FOLDER_ID, "STOP: Output folder-ID secret is missing."

credential_info = json.loads(credentials_text)
credential_type = credential_info.get("type")
if credential_type == "authorized_user":
    credentials = UserCredentials.from_authorized_user_info(
        credential_info,
        scopes=[DRIVE_SCOPE],
    )
elif credential_type == "service_account":
    credentials = service_account.Credentials.from_service_account_info(
        credential_info,
        scopes=[DRIVE_SCOPE],
    )
else:
    raise ValueError(
        "STOP: Drive credential must be authorized_user or service_account JSON."
    )
DRIVE = build("drive", "v3", credentials=credentials, cache_discovery=False)
del credentials_text, credential_info

for folder_label, folder_id, expected_name in (
    ("Dataset A", DATASET_A_FOLDER_ID, "core_army_pro_fans_tweets"),
    ("output", OUTPUT_FOLDER_ID, "TDMEC_PROJECT_OUTPUTS"),
):
    metadata = DRIVE.files().get(
        fileId=folder_id,
        fields="id,name,mimeType,trashed,capabilities(canAddChildren)",
        supportsAllDrives=True,
    ).execute()
    assert metadata["mimeType"] == "application/vnd.google-apps.folder", (
        f"STOP: {folder_label} secret does not identify a Drive folder."
    )
    assert metadata.get("trashed") is not True, (
        f"STOP: {folder_label} folder is trashed."
    )
    assert metadata["name"] == expected_name, (
        f"STOP: {folder_label} folder has an unexpected name."
    )
    if folder_label == "output":
        assert metadata.get("capabilities", {}).get("canAddChildren") is True, (
            "STOP: Drive credentials cannot write persistent output here. "
            "Use authorized_user credentials for My Drive or a writable Shared Drive."
        )

WORK_ROOT = KAGGLE_WORKING_ROOT / "tdmec_dataset_a_node_map"
CACHE_ROOT = WORK_ROOT / "cache"
LOCAL_OUTPUT_ROOT = WORK_ROOT / "outputs"
LOCAL_CANONICAL_NODE_MAP = LOCAL_OUTPUT_ROOT / "node_index_map.parquet"
LOCAL_VALIDATION_MANIFEST = (
    LOCAL_OUTPUT_ROOT / "node_index_map_validation_manifest.json"
)
REPO_ROOT = KAGGLE_WORKING_ROOT / "community-evolution-modeling"
for directory in (WORK_ROOT, CACHE_ROOT, LOCAL_OUTPUT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

print("Authentication method: Kaggle Secret + Google Drive OAuth")
print("Credential type:", credential_type)
print("Local cache root:", CACHE_ROOT)
print("Persistent destination: private Google Drive output folder")
print("PASS: Secure Drive API authentication and storage locations verified.")

## Stage 3 - Clone and pin the audited repository implementation

Repository code is cloned into ephemeral `/kaggle/working` storage. Only the exact ephemeral checkout may be removed on rerun; no Drive file is deleted or moved.

**Stop conditions:** stop on unsafe deletion scope, clone failure, commit mismatch, or a dirty checkout.

In [ ]:
import shutil
import subprocess

REPOSITORY_URL = (
    "https://github.com/faezehmzf/community-evolution-modeling.git"
)
EXPECTED_SHA = "840dd94a80322083bb498a42bbd48fe4cabd85a4"

assert len(EXPECTED_SHA) == 40, "STOP: Audited commit pin is invalid."
assert REPO_ROOT == Path("/kaggle/working/community-evolution-modeling")
assert REPO_ROOT.is_relative_to(KAGGLE_WORKING_ROOT)

# Only this exact ephemeral Kaggle checkout may be recreated on rerun.
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

subprocess.run(
    ["git", "clone", REPOSITORY_URL, str(REPO_ROOT)],
    check=True,
)
subprocess.run(
    ["git", "checkout", "--detach", EXPECTED_SHA],
    cwd=REPO_ROOT,
    check=True,
)
observed_sha = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_ROOT,
    text=True,
).strip()
working_tree_status = subprocess.check_output(
    ["git", "status", "--short"],
    cwd=REPO_ROOT,
    text=True,
).strip()
assert observed_sha == EXPECTED_SHA, "STOP: Repository commit mismatch."
assert working_tree_status == "", "STOP: Repository checkout is dirty."

print("Audited repository commit:", observed_sha)
print("PASS: Repository clone is pinned and clean.")

## Stage 4 - Install and validate imports in the active Kaggle kernel

The supported `test` and `drive` extras are installed with the active Kaggle kernel's `sys.executable`. Standard site-directory processing refreshes a newly created editable-install `.pth` file without manual `sys.path` editing or a runtime restart.

**Stop conditions:** stop on repository-layout mismatch, pip failure, undiscoverable packages, wrong versions, modules outside the pinned checkout, or CLI failure.

In [ ]:
import importlib
import importlib.metadata
import importlib.util
import site
import sys

assert (REPO_ROOT / "pyproject.toml").is_file()
assert (REPO_ROOT / "src").is_dir()
print("sys.executable:", sys.executable)
subprocess.run(
    [sys.executable, "-m", "pip", "--version"],
    cwd=REPO_ROOT,
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[test,drive]"],
    cwd=REPO_ROOT,
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "show", "tdmec-discovery"],
    cwd=REPO_ROOT,
    check=True,
)

site_directories = list(site.getsitepackages())
user_site = site.getusersitepackages()
site_directories.extend([user_site] if isinstance(user_site, str) else user_site)
for directory in site_directories:
    if Path(directory).is_dir():
        site.addsitedir(directory)
importlib.invalidate_caches()

package_names = (
    "tdmec",
    "tdmec_diagnostics",
    "tdmec_discovery",
    "tdmec_pilot",
)
spec_status = {
    name: importlib.util.find_spec(name) is not None
    for name in package_names
}
assert all(spec_status.values()), f"STOP: Package discovery failed: {spec_status}"

import tdmec
import tdmec_diagnostics
import tdmec_discovery
import tdmec_pilot

packages = {
    "tdmec": tdmec,
    "tdmec_diagnostics": tdmec_diagnostics,
    "tdmec_discovery": tdmec_discovery,
    "tdmec_pilot": tdmec_pilot,
}
expected_source_root = (REPO_ROOT / "src").resolve()
for name, package in packages.items():
    package_file = Path(package.__file__).resolve()
    assert package_file.is_relative_to(expected_source_root), (
        f"STOP: {name} resolved outside the pinned checkout."
    )
assert tdmec.__version__ == "0.1.0-phase1"
assert tdmec_diagnostics.__version__ == "0.2.0-phase2"
assert importlib.metadata.version("tdmec-discovery") == "0.1.0"
subprocess.run(
    [sys.executable, "-m", "tdmec_diagnostics.cli", "--help"],
    cwd=REPO_ROOT,
    check=True,
)
print("PASS: Active-kernel installation and imports verified.")

### Stage 4A - Define fail-closed builder helpers over audited APIs

These helpers reuse the pinned repository's `iter_xlsx_rows`, `validate_required_columns`, `parse_user_blob`, `normalize_account_id`, schema constants, `sha256_file`, privacy guard, and later `load_node_map`. They add Drive API transfer, verified per-workbook checkpoints, orchestration, aggregate accounting, structural validation, and atomic publication. No unrelated workbook or identity parser is introduced.

**Stop condition:** stop on any helper-definition import failure.

In [ ]:
import hashlib
import io
import json
import os
import tempfile
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Sequence

import pandas as pd
from pandas.api.types import is_bool_dtype, is_integer_dtype, is_numeric_dtype
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload, MediaIoBaseUpload

from tdmec.hashing import sha256_file
from tdmec_diagnostics.privacy import assert_privacy_safe_mapping
from tdmec_diagnostics.schema_contracts import (
    DATASET_A_ADAPTER_ID,
    DATASET_A_DOCUMENTED_COLUMNS,
    DATASET_A_REQUIRED_COLUMNS,
    DATASET_A_SHEET_NAME,
)
from tdmec_diagnostics.workbook_io import (
    UnsupportedSchemaError,
    iter_xlsx_rows,
    validate_required_columns,
)
from tdmec_pilot.identifiers import normalize_account_id
from tdmec_pilot.user_blob import parse_user_blob

CANONICAL_NODE_MAP_COLUMNS = ("author_account_id", "node_index")
EXPECTED_DATASET_A_FILENAMES = tuple(
    f"core_army_pro_fans_tweets_part_{part:03d}.xlsx"
    for part in range(1, 13)
)

class CanonicalNodeMapError(ValueError):
    pass

class CanonicalNodeMapConflictError(CanonicalNodeMapError):
    pass

@dataclass(frozen=True)
class DatasetAAuthorScan:
    author_ids: frozenset[str]
    workbook_count: int
    total_rows_inspected: int
    valid_author_record_count: int
    missing_author_record_count: int
    malformed_author_record_count: int

@dataclass(frozen=True)
class NodeMapValidation:
    row_count: int
    columns: tuple[str, str]
    index_min: int
    index_max: int
    unique_author_count: int
    unique_index_count: int
    exact_index_set: bool
    canonical_numeric_order: bool

@dataclass(frozen=True)
class CandidateNodeMap:
    path: Path
    sha256: str
    validation: NodeMapValidation

@dataclass(frozen=True)
class PublishedNodeMap:
    path: Path
    sha256: str
    published_new_file: bool
    validation: NodeMapValidation

def discover_dataset_a_workbooks(root: Path) -> list[Path]:
    if not root.is_dir():
        raise FileNotFoundError("Dataset A root is unavailable.")
    workbooks = sorted(root.glob("*.xlsx"), key=lambda path: path.name)
    if tuple(path.name for path in workbooks) != EXPECTED_DATASET_A_FILENAMES:
        raise CanonicalNodeMapError(
            "Dataset A workbook names do not match the canonical set."
        )
    if not all(path.is_file() for path in workbooks):
        raise CanonicalNodeMapError("A Dataset A workbook is not a file.")
    return workbooks

def inspect_dataset_a_workbooks(workbooks: Sequence[Path]) -> None:
    for path in workbooks:
        sheet, header, rows = iter_xlsx_rows(
            path,
            expected_sheet=DATASET_A_SHEET_NAME,
        )
        try:
            if sheet != DATASET_A_SHEET_NAME:
                raise UnsupportedSchemaError("Dataset A worksheet mismatch.")
            validate_required_columns(
                header,
                DATASET_A_REQUIRED_COLUMNS,
                adapter_id=DATASET_A_ADAPTER_ID,
                allow_extra=True,
            )
            if tuple(header) != DATASET_A_DOCUMENTED_COLUMNS:
                raise UnsupportedSchemaError(
                    "Dataset A header is not the documented 31-column schema."
                )
            next(rows, None)
        finally:
            close = getattr(rows, "close", None)
            if close is not None:
                close()

def scan_dataset_a_authors(
    workbooks: Sequence[Path],
    *,
    expected_workbook_count: int = 12,
) -> DatasetAAuthorScan:
    ordered = sorted((Path(path) for path in workbooks), key=lambda path: path.name)
    if len(ordered) != expected_workbook_count:
        raise CanonicalNodeMapError("Unexpected Dataset A workbook count.")
    author_ids = set()
    total_rows = valid = missing = malformed = 0
    for path in ordered:
        sheet, header, rows = iter_xlsx_rows(
            path,
            expected_sheet=DATASET_A_SHEET_NAME,
        )
        if sheet != DATASET_A_SHEET_NAME:
            raise UnsupportedSchemaError("Dataset A worksheet mismatch.")
        columns = validate_required_columns(
            header,
            DATASET_A_REQUIRED_COLUMNS,
            adapter_id=DATASET_A_ADAPTER_ID,
            allow_extra=True,
        )
        if tuple(header) != DATASET_A_DOCUMENTED_COLUMNS:
            raise UnsupportedSchemaError(
                "Dataset A header is not the documented 31-column schema."
            )
        user_column = columns["user"]
        for row in rows:
            total_rows += 1
            raw_user = row[user_column] if user_column < len(row) else None
            parsed = parse_user_blob(raw_user)
            if not parsed.ok:
                if parsed.error == "missing_user":
                    missing += 1
                else:
                    malformed += 1
                continue
            normalized = normalize_account_id(parsed.account_id)
            if normalized is None or normalized != parsed.account_id:
                malformed += 1
                continue
            valid += 1
            author_ids.add(normalized)
    return DatasetAAuthorScan(
        author_ids=frozenset(author_ids),
        workbook_count=len(ordered),
        total_rows_inspected=total_rows,
        valid_author_record_count=valid,
        missing_author_record_count=missing,
        malformed_author_record_count=malformed,
    )

def assert_scan_publishable(
    scan: DatasetAAuthorScan,
    *,
    expected_unique_author_count: int = 16_736,
) -> None:
    if scan.malformed_author_record_count != 0:
        raise CanonicalNodeMapError(
            "Non-null malformed author values prohibit publication."
        )
    if len(scan.author_ids) != expected_unique_author_count:
        raise CanonicalNodeMapError("Unexpected final unique author count.")

def build_canonical_node_map_frame(
    author_ids: Iterable[str],
    *,
    expected_count: int = 16_736,
) -> pd.DataFrame:
    normalized_ids = []
    for value in author_ids:
        if not isinstance(value, str) or not value.isdigit():
            raise CanonicalNodeMapError("Author ID is not an exact digit string.")
        if normalize_account_id(value) != value:
            raise CanonicalNodeMapError("Author ID normalization failed.")
        normalized_ids.append(value)
    unique_ids = set(normalized_ids)
    if len(unique_ids) != expected_count:
        raise CanonicalNodeMapError("Incorrect final node count.")
    ordered_ids = sorted(unique_ids, key=lambda value: (int(value), value))
    frame = pd.DataFrame(
        {
            "author_account_id": pd.Series(ordered_ids, dtype="string"),
            "node_index": pd.Series(range(len(ordered_ids)), dtype="int64"),
        },
        columns=list(CANONICAL_NODE_MAP_COLUMNS),
    )
    validate_canonical_node_map_frame(frame, expected_count=expected_count)
    return frame

def validate_canonical_node_map_frame(
    frame: pd.DataFrame,
    *,
    expected_count: int = 16_736,
) -> NodeMapValidation:
    if tuple(str(column) for column in frame.columns) != CANONICAL_NODE_MAP_COLUMNS:
        raise CanonicalNodeMapError("Incorrect canonical schema or column order.")
    if len(frame) != expected_count:
        raise CanonicalNodeMapError("Incorrect canonical row count.")
    authors = frame["author_account_id"]
    indices = frame["node_index"]
    if authors.isna().any() or indices.isna().any():
        raise CanonicalNodeMapError("Canonical mapping contains null values.")
    if is_numeric_dtype(authors.dtype):
        raise CanonicalNodeMapError("Author dtype is precision-unsafe numeric data.")
    if not authors.map(lambda value: isinstance(value, str)).all():
        raise CanonicalNodeMapError("Author IDs are not stored as strings.")
    if not authors.str.fullmatch(r"\d+").all():
        raise CanonicalNodeMapError("Author ID is not a digit string.")
    if not is_integer_dtype(indices.dtype) or is_bool_dtype(indices.dtype):
        raise CanonicalNodeMapError("node_index is not an integer dtype.")
    unique_authors = int(authors.nunique(dropna=False))
    unique_indices = int(indices.nunique(dropna=False))
    if unique_authors != expected_count:
        raise CanonicalNodeMapError("Duplicate author ID detected.")
    if unique_indices != expected_count:
        raise CanonicalNodeMapError("Duplicate node index detected.")
    integer_indices = indices.astype("int64")
    expected_indices = list(range(expected_count))
    exact_index_set = set(integer_indices.tolist()) == set(expected_indices)
    if not exact_index_set:
        raise CanonicalNodeMapError("Missing or out-of-range node index detected.")
    ordered = frame.assign(node_index=integer_indices).sort_values(
        "node_index",
        kind="mergesort",
    )
    if ordered["node_index"].tolist() != expected_indices:
        raise CanonicalNodeMapError("Canonical index order is incomplete.")
    ordered_authors = ordered["author_account_id"].tolist()
    numeric_authors = sorted(
        ordered_authors,
        key=lambda value: (int(value), value),
    )
    canonical_numeric_order = ordered_authors == numeric_authors
    if not canonical_numeric_order:
        raise CanonicalNodeMapError("Author IDs are not numerically ordered.")
    return NodeMapValidation(
        row_count=len(frame),
        columns=CANONICAL_NODE_MAP_COLUMNS,
        index_min=int(integer_indices.min()),
        index_max=int(integer_indices.max()),
        unique_author_count=unique_authors,
        unique_index_count=unique_indices,
        exact_index_set=exact_index_set,
        canonical_numeric_order=canonical_numeric_order,
    )

def create_candidate_node_map(
    frame: pd.DataFrame,
    private_directory: Path,
    *,
    expected_count: int = 16_736,
) -> CandidateNodeMap:
    validate_canonical_node_map_frame(frame, expected_count=expected_count)
    private_directory.mkdir(parents=True, exist_ok=True)
    handle, name = tempfile.mkstemp(
        prefix=".node-index-map-candidate-",
        suffix=".parquet",
        dir=private_directory,
    )
    os.close(handle)
    candidate_path = Path(name)
    try:
        frame.to_parquet(candidate_path, index=False)
        persisted = pd.read_parquet(candidate_path)
        validation = validate_canonical_node_map_frame(
            persisted,
            expected_count=expected_count,
        )
        return CandidateNodeMap(
            path=candidate_path,
            sha256=sha256_file(candidate_path),
            validation=validation,
        )
    except Exception:
        candidate_path.unlink(missing_ok=True)
        raise

def publish_candidate_node_map(
    candidate: CandidateNodeMap,
    destination: Path,
    *,
    expected_count: int = 16_736,
) -> PublishedNodeMap:
    destination.parent.mkdir(parents=True, exist_ok=True)
    try:
        if not candidate.path.is_file():
            raise CanonicalNodeMapError("Validated candidate is absent.")
        if sha256_file(candidate.path) != candidate.sha256:
            raise CanonicalNodeMapError("Validated candidate checksum changed.")
        candidate_frame = pd.read_parquet(candidate.path)
        candidate_validation = validate_canonical_node_map_frame(
            candidate_frame,
            expected_count=expected_count,
        )
        if destination.exists():
            existing_frame = pd.read_parquet(destination)
            existing_validation = validate_canonical_node_map_frame(
                existing_frame,
                expected_count=expected_count,
            )
            same_mapping = existing_frame.reset_index(drop=True).equals(
                candidate_frame.reset_index(drop=True)
            )
            if not same_mapping:
                raise CanonicalNodeMapConflictError(
                    "Existing canonical map differs and will not be overwritten."
                )
            existing_sha = sha256_file(destination)
            return PublishedNodeMap(
                destination,
                existing_sha,
                False,
                existing_validation,
            )
        os.replace(candidate.path, destination)
        published_sha = sha256_file(destination)
        if published_sha != candidate.sha256:
            raise CanonicalNodeMapError("Published checksum mismatch.")
        published_frame = pd.read_parquet(destination)
        published_validation = validate_canonical_node_map_frame(
            published_frame,
            expected_count=expected_count,
        )
        if not published_frame.reset_index(drop=True).equals(
            candidate_frame.reset_index(drop=True)
        ):
            raise CanonicalNodeMapError("Published mapping mismatch.")
        return PublishedNodeMap(
            destination,
            published_sha,
            True,
            published_validation,
        )
    finally:
        candidate.path.unlink(missing_ok=True)

def require_canonical_node_map(path: Path) -> Path:
    if not path.is_file():
        raise FileNotFoundError("Canonical Dataset A node map is absent.")
    return path

def build_validation_manifest(
    *,
    published: PublishedNodeMap,
    scan: DatasetAAuthorScan,
    audited_repository_commit: str,
    execution_timestamp: str,
) -> dict:
    validation = published.validation
    manifest = {
        "artifact_type": "dataset_a_canonical_node_index_map",
        "canonical_filename": published.path.name,
        "sha256": published.sha256,
        "row_count": validation.row_count,
        "columns": list(validation.columns),
        "index_min": validation.index_min,
        "index_max": validation.index_max,
        "unique_author_count": validation.unique_author_count,
        "unique_index_count": validation.unique_index_count,
        "exact_index_set": validation.exact_index_set,
        "canonical_numeric_order": validation.canonical_numeric_order,
        "dataset_a_workbook_count": scan.workbook_count,
        "total_rows_inspected": scan.total_rows_inspected,
        "valid_author_record_count": scan.valid_author_record_count,
        "missing_author_record_count": scan.missing_author_record_count,
        "malformed_author_record_count": scan.malformed_author_record_count,
        "audited_repository_commit": audited_repository_commit,
        "execution_timestamp": execution_timestamp,
    }
    assert_privacy_safe_mapping(manifest)
    return manifest

def write_validation_manifest_atomic(manifest: dict, destination: Path) -> None:
    assert_privacy_safe_mapping(manifest)
    destination.parent.mkdir(parents=True, exist_ok=True)
    handle, name = tempfile.mkstemp(
        prefix=".node-index-map-manifest-",
        suffix=".json",
        dir=destination.parent,
        text=True,
    )
    temporary_path = Path(name)
    try:
        with os.fdopen(handle, "w", encoding="utf-8") as stream:
            json.dump(manifest, stream, indent=2, sort_keys=True)
            stream.write("\n")
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary_path, destination)
    finally:
        temporary_path.unlink(missing_ok=True)

DRIVE_FILE_FIELDS = (
    "id,name,mimeType,size,md5Checksum,modifiedTime,trashed,parents"
)
CHECKPOINT_PREFIX = ".private_dataset_a_node_map_checkpoint_"
CHECKPOINT_SCHEMA_VERSION = 1
DRIVE_CHUNK_SIZE = 32 * 1024 * 1024

def canonical_json_bytes(value) -> bytes:
    return json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=True,
    ).encode("utf-8")

def sha256_json(value) -> str:
    return hashlib.sha256(canonical_json_bytes(value)).hexdigest()

def md5_file(path: Path) -> str:
    digest = hashlib.md5(usedforsecurity=False)
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def drive_list_children(parent_id: str) -> list[dict]:
    files = []
    page_token = None
    while True:
        response = DRIVE.files().list(
            q=f"'{parent_id}' in parents and trashed = false",
            fields=f"nextPageToken,files({DRIVE_FILE_FIELDS})",
            pageSize=1000,
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        files.extend(response.get("files", []))
        page_token = response.get("nextPageToken")
        if not page_token:
            return files

def drive_find_named(parent_id: str, name: str) -> list[dict]:
    return [item for item in drive_list_children(parent_id) if item["name"] == name]

def ensure_drive_folder(parent_id: str, name: str) -> str:
    matches = drive_find_named(parent_id, name)
    if len(matches) > 1:
        raise CanonicalNodeMapError(f"Multiple Drive folders named {name!r}.")
    if matches:
        if matches[0]["mimeType"] != "application/vnd.google-apps.folder":
            raise CanonicalNodeMapError(f"Drive item {name!r} is not a folder.")
        return matches[0]["id"]
    created = DRIVE.files().create(
        body={
            "name": name,
            "parents": [parent_id],
            "mimeType": "application/vnd.google-apps.folder",
        },
        fields="id",
        supportsAllDrives=True,
    ).execute()
    return created["id"]

def drive_download_bytes(file_id: str) -> bytes:
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(
        buffer,
        DRIVE.files().get_media(fileId=file_id, supportsAllDrives=True),
        chunksize=DRIVE_CHUNK_SIZE,
    )
    done = False
    while not done:
        _, done = downloader.next_chunk(num_retries=5)
    return buffer.getvalue()

def drive_download_file(metadata: dict, destination: Path) -> Path:
    expected_size = int(metadata["size"])
    expected_md5 = metadata.get("md5Checksum")
    if not expected_md5:
        raise CanonicalNodeMapError("Drive file lacks an MD5 checksum.")
    if destination.is_file():
        if destination.stat().st_size == expected_size and md5_file(destination) == expected_md5:
            return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_suffix(destination.suffix + ".part")
    partial.unlink(missing_ok=True)
    with partial.open("wb") as stream:
        downloader = MediaIoBaseDownload(
            stream,
            DRIVE.files().get_media(
                fileId=metadata["id"],
                supportsAllDrives=True,
            ),
            chunksize=DRIVE_CHUNK_SIZE,
        )
        done = False
        while not done:
            _, done = downloader.next_chunk(num_retries=5)
    if partial.stat().st_size != expected_size or md5_file(partial) != expected_md5:
        partial.unlink(missing_ok=True)
        raise CanonicalNodeMapError("Downloaded Drive file checksum mismatch.")
    os.replace(partial, destination)
    return destination

def execute_resumable_upload(request) -> dict:
    response = None
    while response is None:
        _, response = request.next_chunk(num_retries=5)
    return response

def drive_create_bytes_once(
    parent_id: str,
    name: str,
    payload: bytes,
    mime_type: str,
) -> dict:
    matches = drive_find_named(parent_id, name)
    if len(matches) > 1:
        raise CanonicalNodeMapError(f"Multiple Drive files named {name!r}.")
    if matches:
        if drive_download_bytes(matches[0]["id"]) != payload:
            raise CanonicalNodeMapConflictError(
                f"Existing persistent file {name!r} differs."
            )
        return matches[0]
    media = MediaIoBaseUpload(
        io.BytesIO(payload),
        mimetype=mime_type,
        chunksize=DRIVE_CHUNK_SIZE,
        resumable=True,
    )
    request = DRIVE.files().create(
        body={"name": name, "parents": [parent_id]},
        media_body=media,
        fields=DRIVE_FILE_FIELDS,
        supportsAllDrives=True,
    )
    return execute_resumable_upload(request)

def drive_replace_json(parent_id: str, name: str, payload: dict) -> dict:
    serialized = json.dumps(payload, indent=2, sort_keys=True).encode("utf-8") + b"\n"
    matches = drive_find_named(parent_id, name)
    if len(matches) > 1:
        raise CanonicalNodeMapError(f"Multiple Drive files named {name!r}.")
    media = MediaIoBaseUpload(
        io.BytesIO(serialized),
        mimetype="application/json",
        chunksize=DRIVE_CHUNK_SIZE,
        resumable=True,
    )
    if matches:
        request = DRIVE.files().update(
            fileId=matches[0]["id"],
            media_body=media,
            fields=DRIVE_FILE_FIELDS,
            supportsAllDrives=True,
        )
    else:
        request = DRIVE.files().create(
            body={"name": name, "parents": [parent_id]},
            media_body=media,
            fields=DRIVE_FILE_FIELDS,
            supportsAllDrives=True,
        )
    return execute_resumable_upload(request)

def drive_upload_file_new(parent_id: str, name: str, local_path: Path) -> dict:
    if drive_find_named(parent_id, name):
        raise CanonicalNodeMapConflictError(
            f"Drive destination {name!r} appeared before publication."
        )
    media = MediaFileUpload(
        str(local_path),
        mimetype="application/octet-stream",
        chunksize=DRIVE_CHUNK_SIZE,
        resumable=True,
    )
    request = DRIVE.files().create(
        body={"name": name, "parents": [parent_id]},
        media_body=media,
        fields=DRIVE_FILE_FIELDS,
        supportsAllDrives=True,
    )
    return execute_resumable_upload(request)

def checkpoint_name(completed_count: int, parser_hash: str, inventory_hash: str) -> str:
    return (
        f"{CHECKPOINT_PREFIX}{completed_count:03d}_"
        f"{parser_hash[:12]}_{inventory_hash[:12]}.json"
    )

def build_scan_checkpoint(
    *,
    author_ids: set[str],
    completed_files: list[dict],
    input_inventory: list[dict],
    parser_contract: dict,
    parser_hash: str,
    inventory_hash: str,
) -> dict:
    checkpoint = {
        "checkpoint_schema_version": CHECKPOINT_SCHEMA_VERSION,
        "audited_repository_commit": EXPECTED_SHA,
        "parser_contract": parser_contract,
        "parser_config_hash": parser_hash,
        "input_inventory_hash": inventory_hash,
        "input_inventory": input_inventory,
        "completed_count": len(completed_files),
        "completed_files": completed_files,
        "author_ids": sorted(author_ids, key=lambda value: (int(value), value)),
        "total_rows_inspected": sum(item["rows_inspected"] for item in completed_files),
        "valid_author_record_count": sum(item["valid_author_records"] for item in completed_files),
        "missing_author_record_count": sum(item["missing_author_records"] for item in completed_files),
        "malformed_author_record_count": sum(item["malformed_author_records"] for item in completed_files),
    }
    checkpoint["checkpoint_content_sha256"] = sha256_json(checkpoint)
    return checkpoint

def validate_scan_checkpoint(
    checkpoint: dict,
    *,
    input_inventory: list[dict],
    parser_contract: dict,
    parser_hash: str,
    inventory_hash: str,
) -> None:
    expected_keys = {
        "checkpoint_schema_version",
        "checkpoint_content_sha256",
        "audited_repository_commit",
        "parser_contract",
        "parser_config_hash",
        "input_inventory_hash",
        "input_inventory",
        "completed_count",
        "completed_files",
        "author_ids",
        "total_rows_inspected",
        "valid_author_record_count",
        "missing_author_record_count",
        "malformed_author_record_count",
    }
    if set(checkpoint) != expected_keys:
        raise CanonicalNodeMapError("Checkpoint fields are invalid.")
    unsigned_checkpoint = {
        key: value
        for key, value in checkpoint.items()
        if key != "checkpoint_content_sha256"
    }
    if checkpoint["checkpoint_content_sha256"] != sha256_json(
        unsigned_checkpoint
    ):
        raise CanonicalNodeMapError("Checkpoint content hash mismatch.")
    if checkpoint["checkpoint_schema_version"] != CHECKPOINT_SCHEMA_VERSION:
        raise CanonicalNodeMapError("Checkpoint schema version mismatch.")
    if checkpoint["audited_repository_commit"] != EXPECTED_SHA:
        raise CanonicalNodeMapError("Checkpoint repository commit mismatch.")
    if checkpoint["parser_contract"] != parser_contract:
        raise CanonicalNodeMapError("Checkpoint parser contract mismatch.")
    if checkpoint["parser_config_hash"] != parser_hash:
        raise CanonicalNodeMapError("Checkpoint parser hash mismatch.")
    if checkpoint["input_inventory_hash"] != inventory_hash:
        raise CanonicalNodeMapError("Checkpoint input inventory hash mismatch.")
    if checkpoint["input_inventory"] != input_inventory:
        raise CanonicalNodeMapError("Checkpoint input inventory mismatch.")
    completed = checkpoint["completed_files"]
    count = checkpoint["completed_count"]
    if not isinstance(count, int) or count != len(completed) or not 0 <= count <= 12:
        raise CanonicalNodeMapError("Checkpoint completed-file count is invalid.")
    if [item["name"] for item in completed] != [item["name"] for item in input_inventory[:count]]:
        raise CanonicalNodeMapError("Checkpoint completed files are not a strict prefix.")
    for completed_item, inventory_item in zip(completed, input_inventory):
        for key in ("id", "name", "size", "md5Checksum"):
            if completed_item[key] != inventory_item[key]:
                raise CanonicalNodeMapError("Checkpoint file identity mismatch.")
    author_ids = checkpoint["author_ids"]
    if len(author_ids) != len(set(author_ids)):
        raise CanonicalNodeMapError("Checkpoint contains duplicate author IDs.")
    if not all(isinstance(value, str) and value.isdigit() for value in author_ids):
        raise CanonicalNodeMapError("Checkpoint contains a malformed author ID.")
    aggregate_fields = {
        "total_rows_inspected": "rows_inspected",
        "valid_author_record_count": "valid_author_records",
        "missing_author_record_count": "missing_author_records",
        "malformed_author_record_count": "malformed_author_records",
    }
    for aggregate_key, item_key in aggregate_fields.items():
        if checkpoint[aggregate_key] != sum(item[item_key] for item in completed):
            raise CanonicalNodeMapError("Checkpoint aggregate count mismatch.")

def save_scan_checkpoint(
    checkpoint_folder_id: str,
    checkpoint: dict,
) -> dict:
    name = checkpoint_name(
        checkpoint["completed_count"],
        checkpoint["parser_config_hash"],
        checkpoint["input_inventory_hash"],
    )
    return drive_create_bytes_once(
        checkpoint_folder_id,
        name,
        canonical_json_bytes(checkpoint),
        "application/json",
    )

def load_latest_scan_checkpoint(
    checkpoint_folder_id: str,
    *,
    input_inventory: list[dict],
    parser_contract: dict,
    parser_hash: str,
    inventory_hash: str,
) -> dict | None:
    suffix = f"_{parser_hash[:12]}_{inventory_hash[:12]}.json"
    candidates = sorted(
        (
            item
            for item in drive_list_children(checkpoint_folder_id)
            if item["name"].startswith(CHECKPOINT_PREFIX)
            and item["name"].endswith(suffix)
        ),
        key=lambda item: item["name"],
        reverse=True,
    )
    if not candidates:
        return None
    try:
        checkpoint = json.loads(drive_download_bytes(candidates[0]["id"]))
    except Exception as exc:
        raise CanonicalNodeMapError("Latest persistent checkpoint is unreadable.") from exc
    validate_scan_checkpoint(
        checkpoint,
        input_inventory=input_inventory,
        parser_contract=parser_contract,
        parser_hash=parser_hash,
        inventory_hash=inventory_hash,
    )
    return checkpoint

print("PASS: Fail-closed builder helpers loaded over audited repository APIs.")

### Stage 4B - Run synthetic fail-closed preflight

This preflight uses only temporary synthetic data. It checks deterministic numeric ordering, exact-string tie-breaking, duplicate authors, duplicate indices, missing interior indices, malformed authors, incorrect final count, exact schema, checkpoint validation and tamper rejection, identical mappings with identical or different valid Parquet bytes, conflicting mappings, preservation of the accepted existing checksum, and the absent-canonical hard stop.

**Stop condition:** stop on any failed synthetic assertion before reading Dataset A.

In [ ]:
import tempfile

import openpyxl

def synthetic_frame(authors, indices):
    return pd.DataFrame(
        {
            "author_account_id": pd.Series(authors, dtype="string"),
            "node_index": pd.Series(indices, dtype="int64"),
        },
        columns=list(CANONICAL_NODE_MAP_COLUMNS),
    )

def expect_error(error_type, action):
    try:
        action()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

numeric = build_canonical_node_map_frame(["10", "2", "1"], expected_count=3)
assert numeric["author_account_id"].tolist() == ["1", "2", "10"]
tied = build_canonical_node_map_frame(["2", "1", "01"], expected_count=3)
assert tied["author_account_id"].tolist() == ["01", "1", "2"]
expect_error(
    CanonicalNodeMapError,
    lambda: validate_canonical_node_map_frame(
        synthetic_frame(["1", "1", "2"], [0, 1, 2]),
        expected_count=3,
    ),
)
expect_error(
    CanonicalNodeMapError,
    lambda: validate_canonical_node_map_frame(
        synthetic_frame(["1", "2", "3"], [0, 0, 2]),
        expected_count=3,
    ),
)
expect_error(
    CanonicalNodeMapError,
    lambda: validate_canonical_node_map_frame(
        synthetic_frame(["1", "2", "3"], [0, 2, 3]),
        expected_count=3,
    ),
)
expect_error(
    CanonicalNodeMapError,
    lambda: build_canonical_node_map_frame(["1", "2"], expected_count=3),
)
for invalid in (
    numeric[["node_index", "author_account_id"]],
    numeric.assign(extra=0),
    numeric[["author_account_id"]],
):
    expect_error(
        CanonicalNodeMapError,
        lambda invalid=invalid: validate_canonical_node_map_frame(
            invalid,
            expected_count=3,
        ),
    )

with tempfile.TemporaryDirectory() as temporary_directory:
    temporary_root = Path(temporary_directory)
    workbook_path = temporary_root / "synthetic.xlsx"
    workbook = openpyxl.Workbook()
    worksheet = workbook.active
    worksheet.title = "tweets"
    worksheet.append(list(DATASET_A_DOCUMENTED_COLUMNS))
    user_column = DATASET_A_DOCUMENTED_COLUMNS.index("user")
    for user_value in ("{'id': 10}", None, "not-a-user"):
        row = [None] * len(DATASET_A_DOCUMENTED_COLUMNS)
        row[user_column] = user_value
        worksheet.append(row)
    workbook.save(workbook_path)
    scan = scan_dataset_a_authors(
        [workbook_path],
        expected_workbook_count=1,
    )
    assert scan.valid_author_record_count == 1
    assert scan.missing_author_record_count == 1
    assert scan.malformed_author_record_count == 1
    expect_error(
        CanonicalNodeMapError,
        lambda: assert_scan_publishable(scan, expected_unique_author_count=1),
    )

    synthetic_inventory = [
        {
            "id": "synthetic-file-id",
            "name": "synthetic.xlsx",
            "size": 123,
            "md5Checksum": "0" * 32,
        }
    ]
    synthetic_parser_contract = {
        "sheet": "tweets",
        "columns": list(DATASET_A_DOCUMENTED_COLUMNS),
        "user_parser": "tdmec_pilot.user_blob.parse_user_blob",
        "account_normalizer": "tdmec_pilot.identifiers.normalize_account_id",
    }
    synthetic_parser_hash = sha256_json(synthetic_parser_contract)
    synthetic_inventory_hash = sha256_json(synthetic_inventory)
    completed_file = {
        **synthetic_inventory[0],
        "rows_inspected": 1,
        "valid_author_records": 1,
        "missing_author_records": 0,
        "malformed_author_records": 0,
    }
    synthetic_checkpoint = build_scan_checkpoint(
        author_ids={"10"},
        completed_files=[completed_file],
        input_inventory=synthetic_inventory,
        parser_contract=synthetic_parser_contract,
        parser_hash=synthetic_parser_hash,
        inventory_hash=synthetic_inventory_hash,
    )
    validate_scan_checkpoint(
        synthetic_checkpoint,
        input_inventory=synthetic_inventory,
        parser_contract=synthetic_parser_contract,
        parser_hash=synthetic_parser_hash,
        inventory_hash=synthetic_inventory_hash,
    )
    tampered_checkpoint = dict(synthetic_checkpoint)
    tampered_checkpoint["input_inventory_hash"] = "f" * 64
    expect_error(
        CanonicalNodeMapError,
        lambda: validate_scan_checkpoint(
            tampered_checkpoint,
            input_inventory=synthetic_inventory,
            parser_contract=synthetic_parser_contract,
            parser_hash=synthetic_parser_hash,
            inventory_hash=synthetic_inventory_hash,
        ),
    )

    canonical_path = temporary_root / "node_index_map.parquet"
    first = publish_candidate_node_map(
        create_candidate_node_map(numeric, temporary_root, expected_count=3),
        canonical_path,
        expected_count=3,
    )
    identical_candidate = create_candidate_node_map(
        numeric,
        temporary_root,
        expected_count=3,
    )
    assert identical_candidate.sha256 == first.sha256
    second = publish_candidate_node_map(
        identical_candidate,
        canonical_path,
        expected_count=3,
    )
    assert first.published_new_file is True
    assert second.published_new_file is False
    assert first.sha256 == second.sha256

    numeric.to_parquet(canonical_path, index=False, compression="gzip")
    different_existing_bytes = canonical_path.read_bytes()
    different_existing_sha = sha256_file(canonical_path)
    assert different_existing_sha != first.sha256
    different_candidate = create_candidate_node_map(
        numeric,
        temporary_root,
        expected_count=3,
    )
    assert different_candidate.sha256 != different_existing_sha
    accepted_different_bytes = publish_candidate_node_map(
        different_candidate,
        canonical_path,
        expected_count=3,
    )
    assert accepted_different_bytes.published_new_file is False
    assert accepted_different_bytes.sha256 == different_existing_sha
    assert canonical_path.read_bytes() == different_existing_bytes

    synthetic_scan = DatasetAAuthorScan(
        author_ids=frozenset({"1", "2", "10"}),
        workbook_count=1,
        total_rows_inspected=3,
        valid_author_record_count=3,
        missing_author_record_count=0,
        malformed_author_record_count=0,
    )
    synthetic_manifest = build_validation_manifest(
        published=accepted_different_bytes,
        scan=synthetic_scan,
        audited_repository_commit="0" * 40,
        execution_timestamp="2000-01-01T00:00:00Z",
    )
    assert synthetic_manifest["sha256"] == different_existing_sha

    original_bytes = canonical_path.read_bytes()
    conflicting = build_canonical_node_map_frame(
        ["1", "2", "4"],
        expected_count=3,
    )
    expect_error(
        CanonicalNodeMapConflictError,
        lambda: publish_candidate_node_map(
            create_candidate_node_map(
                conflicting,
                temporary_root,
                expected_count=3,
            ),
            canonical_path,
            expected_count=3,
        ),
    )
    assert canonical_path.read_bytes() == original_bytes
    expect_error(
        FileNotFoundError,
        lambda: require_canonical_node_map(temporary_root / "absent.parquet"),
    )

print("PASS: Embedded synthetic canonical node-map preflight succeeded.")

## Stage 5 - Discover and fingerprint exactly 12 Dataset A workbooks

The Drive API lists only direct children of the configured Dataset A folder. Discovery requires the exact documented filenames from part 001 through part 012, a byte size, and a Drive MD5 checksum for every workbook. The resulting inventory hash binds every persistent checkpoint to this exact input set. No workbook content, Drive file ID, or author identifier is printed.

**Stop conditions:** stop on a missing, extra, renamed, duplicated, non-XLSX, size-less, or checksum-less workbook entry.

In [ ]:
MANIFESTS_FOLDER_ID = ensure_drive_folder(OUTPUT_FOLDER_ID, "manifests")
CHECKPOINTS_FOLDER_ID = ensure_drive_folder(OUTPUT_FOLDER_ID, "checkpoints")

# Use Drive metadata for deterministic discovery without downloading rows.
dataset_children = drive_list_children(DATASET_A_FOLDER_ID)
DATASET_A_REMOTE_FILES = sorted(
    (
        item
        for item in dataset_children
        if item["mimeType"] != "application/vnd.google-apps.folder"
        and item["name"].lower().endswith(".xlsx")
    ),
    key=lambda item: item["name"],
)
assert len(DATASET_A_REMOTE_FILES) == 12, (
    "STOP: Dataset A must contain exactly 12 direct XLSX files."
)
assert tuple(item["name"] for item in DATASET_A_REMOTE_FILES) == (
    EXPECTED_DATASET_A_FILENAMES
), "STOP: Dataset A workbook names do not match the canonical set."
assert all(item.get("size") for item in DATASET_A_REMOTE_FILES), (
    "STOP: A Dataset A workbook lacks a Drive byte size."
)
assert all(item.get("md5Checksum") for item in DATASET_A_REMOTE_FILES), (
    "STOP: A Dataset A workbook lacks a Drive MD5 checksum."
)
INPUT_INVENTORY = [
    {
        "id": item["id"],
        "name": item["name"],
        "size": int(item["size"]),
        "md5Checksum": item["md5Checksum"],
    }
    for item in DATASET_A_REMOTE_FILES
]
INPUT_INVENTORY_HASH = sha256_json(INPUT_INVENTORY)
PARSER_CONTRACT = {
    "worksheet": DATASET_A_SHEET_NAME,
    "documented_columns": list(DATASET_A_DOCUMENTED_COLUMNS),
    "required_columns": list(DATASET_A_REQUIRED_COLUMNS),
    "row_reader": "tdmec_diagnostics.workbook_io.iter_xlsx_rows",
    "schema_validator": (
        "tdmec_diagnostics.workbook_io.validate_required_columns"
    ),
    "user_parser": "tdmec_pilot.user_blob.parse_user_blob",
    "account_normalizer": (
        "tdmec_pilot.identifiers.normalize_account_id"
    ),
}
PARSER_CONFIG_HASH = sha256_json(
    {"repository_commit": EXPECTED_SHA, "contract": PARSER_CONTRACT}
)
print("Dataset A workbook count:", len(DATASET_A_REMOTE_FILES))
print("Input inventory SHA-256:", INPUT_INVENTORY_HASH)
print("Parser configuration SHA-256:", PARSER_CONFIG_HASH)
print("PASS: Exact checksummed Dataset A inventory discovered.")

## Stage 6 - Load a validated persistent completed-workbook checkpoint

A checkpoint is reused only when it matches the audited commit, parser contract and hash, complete Drive inventory and hash, and a strict prefix of completed workbooks. Each checkpoint is immutable and contains the normalized author set needed to continue scientifically exact aggregation. Because those IDs are private, checkpoints stay only in the configured private Google Drive output folder and are never printed.

If no matching checkpoint exists, processing starts at workbook 1. Checkpoints created for another commit, parser, or input inventory are ignored by filename; a malformed latest matching checkpoint causes a hard stop rather than an unsafe restart.

**Stop conditions:** stop on an unreadable, tampered, internally inconsistent, or non-prefix matching checkpoint.

In [ ]:
LATEST_CHECKPOINT = load_latest_scan_checkpoint(
    CHECKPOINTS_FOLDER_ID,
    input_inventory=INPUT_INVENTORY,
    parser_contract=PARSER_CONTRACT,
    parser_hash=PARSER_CONFIG_HASH,
    inventory_hash=INPUT_INVENTORY_HASH,
)
if LATEST_CHECKPOINT is None:
    all_author_ids = set()
    completed_files = []
    total_rows_inspected = 0
    valid_author_record_count = 0
    missing_author_record_count = 0
    malformed_author_record_count = 0
else:
    all_author_ids = set(LATEST_CHECKPOINT["author_ids"])
    completed_files = list(LATEST_CHECKPOINT["completed_files"])
    total_rows_inspected = LATEST_CHECKPOINT["total_rows_inspected"]
    valid_author_record_count = LATEST_CHECKPOINT[
        "valid_author_record_count"
    ]
    missing_author_record_count = LATEST_CHECKPOINT[
        "missing_author_record_count"
    ]
    malformed_author_record_count = LATEST_CHECKPOINT[
        "malformed_author_record_count"
    ]

RESUMED_COMPLETED_COUNT = len(completed_files)
print("Completed workbooks restored:", RESUMED_COMPLETED_COUNT)
print("Rows restored:", total_rows_inspected)
print("PASS: Persistent checkpoint state is valid and ready.")

## Stage 7 - Stream every Dataset A row with the audited parser

Each uncompleted workbook is downloaded to ephemeral Kaggle storage with verified Drive size and MD5, then streamed separately using `iter_xlsx_rows`, `validate_required_columns`, `parse_user_blob`, and `normalize_account_id`. Worksheet and exact 31-column schema validation happen inside the audited scan before row processing. Missing and non-null malformed author records are counted separately.

After a whole workbook finishes, one immutable checkpoint containing the complete aggregate state is uploaded. Only then is its ephemeral XLSX cache deleted. A session interruption can therefore repeat at most the current workbook; it cannot double-count a completed workbook. No row-level resume is claimed.

**Stop conditions:** stop on a download checksum mismatch, parser or schema exception, malformed author value, or checkpoint upload conflict. A workbook is never marked complete before its full scan succeeds.

In [ ]:
ACTIVE_CHECKPOINT = load_latest_scan_checkpoint(
    CHECKPOINTS_FOLDER_ID,
    input_inventory=INPUT_INVENTORY,
    parser_contract=PARSER_CONTRACT,
    parser_hash=PARSER_CONFIG_HASH,
    inventory_hash=INPUT_INVENTORY_HASH,
)
if ACTIVE_CHECKPOINT is None:
    all_author_ids = set()
    completed_files = []
    total_rows_inspected = 0
    valid_author_record_count = 0
    missing_author_record_count = 0
    malformed_author_record_count = 0
else:
    all_author_ids = set(ACTIVE_CHECKPOINT["author_ids"])
    completed_files = list(ACTIVE_CHECKPOINT["completed_files"])
    total_rows_inspected = ACTIVE_CHECKPOINT["total_rows_inspected"]
    valid_author_record_count = ACTIVE_CHECKPOINT[
        "valid_author_record_count"
    ]
    missing_author_record_count = ACTIVE_CHECKPOINT[
        "missing_author_record_count"
    ]
    malformed_author_record_count = ACTIVE_CHECKPOINT[
        "malformed_author_record_count"
    ]
RESUMED_COMPLETED_COUNT = len(completed_files)

for workbook_number, remote_file in enumerate(DATASET_A_REMOTE_FILES, start=1):
    if workbook_number <= RESUMED_COMPLETED_COUNT:
        continue
    workbook_path = CACHE_ROOT / remote_file["name"]
    drive_download_file(remote_file, workbook_path)
    inspect_dataset_a_workbooks([workbook_path])
    partial_scan = scan_dataset_a_authors(
        [workbook_path],
        expected_workbook_count=1,
    )
    all_author_ids.update(partial_scan.author_ids)
    total_rows_inspected += partial_scan.total_rows_inspected
    valid_author_record_count += partial_scan.valid_author_record_count
    missing_author_record_count += partial_scan.missing_author_record_count
    malformed_author_record_count += partial_scan.malformed_author_record_count
    completed_files.append(
        {
            "id": remote_file["id"],
            "name": remote_file["name"],
            "size": int(remote_file["size"]),
            "md5Checksum": remote_file["md5Checksum"],
            "rows_inspected": partial_scan.total_rows_inspected,
            "valid_author_records": partial_scan.valid_author_record_count,
            "missing_author_records": partial_scan.missing_author_record_count,
            "malformed_author_records": (
                partial_scan.malformed_author_record_count
            ),
        }
    )
    # Seal state only after the entire checksum-verified workbook is parsed.
    checkpoint = build_scan_checkpoint(
        author_ids=all_author_ids,
        completed_files=completed_files,
        input_inventory=INPUT_INVENTORY,
        parser_contract=PARSER_CONTRACT,
        parser_hash=PARSER_CONFIG_HASH,
        inventory_hash=INPUT_INVENTORY_HASH,
    )
    save_scan_checkpoint(CHECKPOINTS_FOLDER_ID, checkpoint)
    workbook_path.unlink(missing_ok=True)
    print(
        f"Processed workbook {workbook_number}/12; "
        f"rows inspected so far: {total_rows_inspected}"
    )
    if partial_scan.malformed_author_record_count:
        raise CanonicalNodeMapError(
            "STOP: Non-null malformed author values prohibit continuation."
        )

DATASET_A_SCAN = DatasetAAuthorScan(
    author_ids=frozenset(all_author_ids),
    workbook_count=len(completed_files),
    total_rows_inspected=total_rows_inspected,
    valid_author_record_count=valid_author_record_count,
    missing_author_record_count=missing_author_record_count,
    malformed_author_record_count=malformed_author_record_count,
)
print("Total rows inspected:", DATASET_A_SCAN.total_rows_inspected)
print("Valid author records:", DATASET_A_SCAN.valid_author_record_count)
print("Missing author records:", DATASET_A_SCAN.missing_author_record_count)
print("Malformed author records:", DATASET_A_SCAN.malformed_author_record_count)
assert DATASET_A_SCAN.workbook_count == 12
print("PASS: Complete Dataset A streaming scan and checkpointing finished.")

## Stage 8 - Validate normalized Dataset A author identities

All retained author IDs must already be exact digit strings produced by the audited parser and normalizer. Non-null malformed author records prohibit publication. The distinct author universe must contain exactly 16,736 IDs.

**Stop conditions:** stop on any malformed non-null author value, non-digit normalized ID, or final distinct-author count other than 16,736.

In [ ]:
assert_scan_publishable(
    DATASET_A_SCAN,
    expected_unique_author_count=16_736,
)
assert all(
    isinstance(author_id, str) and author_id.isdigit()
    for author_id in DATASET_A_SCAN.author_ids
)
print("Unique normalized Dataset A authors:", len(DATASET_A_SCAN.author_ids))
print("PASS: Dataset A author universe is publishable.")

## Stage 9 - Build the deterministic two-column mapping

Distinct exact author strings are sorted by exact integer value. Ascending positions become node indices 0 through 16,735. The resulting DataFrame contains only `author_account_id` and `node_index` in that order.

**Stop conditions:** stop on a malformed ID, unexpected unique count, or mapping-construction failure.

In [ ]:
CANONICAL_FRAME = build_canonical_node_map_frame(
    DATASET_A_SCAN.author_ids,
    expected_count=16_736,
)
assert CANONICAL_FRAME.columns.tolist() == list(CANONICAL_NODE_MAP_COLUMNS)
assert len(CANONICAL_FRAME) == 16_736
print("Canonical row count:", len(CANONICAL_FRAME))
print("Canonical columns:", CANONICAL_FRAME.columns.tolist())
print("PASS: Deterministic canonical mapping built.")

## Stage 10 - Validate every structural invariant in memory

Validation covers exact schema and order, precision-safe string authors, integer indices, null exclusion, uniqueness, the complete index set, and numeric author ordering. No row or identifier is printed.

**Stop conditions:** stop on any failed structural invariant.

In [ ]:
IN_MEMORY_VALIDATION = validate_canonical_node_map_frame(
    CANONICAL_FRAME,
    expected_count=16_736,
)
assert IN_MEMORY_VALIDATION.row_count == 16_736
assert IN_MEMORY_VALIDATION.index_min == 0
assert IN_MEMORY_VALIDATION.index_max == 16_735
assert IN_MEMORY_VALIDATION.unique_author_count == 16_736
assert IN_MEMORY_VALIDATION.unique_index_count == 16_736
assert IN_MEMORY_VALIDATION.exact_index_set is True
assert IN_MEMORY_VALIDATION.canonical_numeric_order is True
print("PASS: In-memory canonical mapping invariants verified.")

## Stage 11 - Create and hash a fully validated private candidate

Only after the complete scan and in-memory validation pass is a temporary Parquet candidate written under ephemeral Kaggle working storage. The candidate is read back, revalidated, and hashed with streaming SHA-256. It is not yet the canonical Drive file.

**Stop conditions:** stop on candidate write, read-back, schema, invariant, or checksum failure.

In [ ]:
NODE_MAP_CANDIDATE = create_candidate_node_map(
    CANONICAL_FRAME,
    LOCAL_OUTPUT_ROOT,
    expected_count=16_736,
)
assert NODE_MAP_CANDIDATE.path.parent == LOCAL_OUTPUT_ROOT
assert NODE_MAP_CANDIDATE.path.is_file()
assert len(NODE_MAP_CANDIDATE.sha256) == 64
print("Candidate row count:", NODE_MAP_CANDIDATE.validation.row_count)
print("Candidate SHA-256:", NODE_MAP_CANDIDATE.sha256)
print("PASS: Complete ephemeral candidate validated and hashed.")

## Stage 12 - Publish to persistent Drive or accept an identical mapping

The canonical destination is the exact `manifests/node_index_map.parquet` child of the configured private Drive output folder. If it already exists, it is downloaded with size and MD5 verification, structurally validated, and compared with the complete rebuilt mapping. An equal mapping is accepted without overwrite even when Parquet bytes differ, preserving the existing file's SHA-256. A different mapping hard-stops.

For first publication, candidate checksum stability is verified before a resumable Drive upload. The completed remote object is downloaded again and its SHA-256 and mapping are revalidated. An interrupted upload is never accepted as canonical.

**Stop conditions:** stop on duplicate Drive destinations, a conflicting pre-existing mapping, transfer checksum drift, content drift, or incomplete publication.

In [ ]:
CANONICAL_FILENAME = "node_index_map.parquet"
remote_matches = drive_find_named(MANIFESTS_FOLDER_ID, CANONICAL_FILENAME)
assert len(remote_matches) <= 1, (
    "STOP: Multiple canonical node maps exist in the Drive manifests folder."
)
# Existing Drive content is downloaded and compared; it is never overwritten.
if remote_matches:
    REMOTE_CANONICAL_FILE = remote_matches[0]
    assert REMOTE_CANONICAL_FILE["mimeType"] != (
        "application/vnd.google-apps.folder"
    ), "STOP: Canonical Drive destination is a folder."
    drive_download_file(REMOTE_CANONICAL_FILE, LOCAL_CANONICAL_NODE_MAP)
else:
    LOCAL_CANONICAL_NODE_MAP.unlink(missing_ok=True)

PUBLISHED_NODE_MAP = publish_candidate_node_map(
    NODE_MAP_CANDIDATE,
    LOCAL_CANONICAL_NODE_MAP,
    expected_count=16_736,
)
assert PUBLISHED_NODE_MAP.path == LOCAL_CANONICAL_NODE_MAP
assert PUBLISHED_NODE_MAP.path.is_file()

if remote_matches:
    assert PUBLISHED_NODE_MAP.published_new_file is False
    assert sha256_file(LOCAL_CANONICAL_NODE_MAP) == PUBLISHED_NODE_MAP.sha256
else:
    assert PUBLISHED_NODE_MAP.published_new_file is True
    REMOTE_CANONICAL_FILE = drive_upload_file_new(
        MANIFESTS_FOLDER_ID,
        CANONICAL_FILENAME,
        LOCAL_CANONICAL_NODE_MAP,
    )
    verification_path = LOCAL_OUTPUT_ROOT / ".published-node-map-check.parquet"
    verification_path.unlink(missing_ok=True)
    drive_download_file(REMOTE_CANONICAL_FILE, verification_path)
    try:
        assert sha256_file(verification_path) == PUBLISHED_NODE_MAP.sha256, (
            "STOP: Published Drive SHA-256 differs from the candidate."
        )
        verification_frame = pd.read_parquet(verification_path)
        validate_canonical_node_map_frame(
            verification_frame,
            expected_count=16_736,
        )
        assert verification_frame.reset_index(drop=True).equals(
            CANONICAL_FRAME.reset_index(drop=True)
        ), "STOP: Published Drive mapping differs from the candidate."
    finally:
        verification_path.unlink(missing_ok=True)

print("Published new canonical file:", PUBLISHED_NODE_MAP.published_new_file)
print("Canonical SHA-256:", PUBLISHED_NODE_MAP.sha256)
print("PASS: Persistent canonical node map published or accepted safely.")

## Stage 13 - Validate the published artifact with `load_node_map`

The repository's actual `load_node_map(...) -> NodeMap` interface verifies the expected count and outer index bounds. `NodeMap` is a dataclass, not a DataFrame.

**Stop conditions:** stop on any loader exception, wrong return type, wrong count, or wrong index bounds.

In [ ]:
from tdmec_pilot.node_map import NodeMap, load_node_map

LOADED_NODE_MAP = load_node_map(
    LOCAL_CANONICAL_NODE_MAP,
    expected_count=16_736,
    index_min=0,
    index_max=16_735,
)
assert isinstance(LOADED_NODE_MAP, NodeMap)
assert len(LOADED_NODE_MAP) == 16_736
assert LOADED_NODE_MAP.min_index == 0
assert LOADED_NODE_MAP.max_index == 16_735
print("PASS: Repository NodeMap loader validation succeeded.")

## Stage 14 - Perform supplemental published-artifact validation

The published Parquet file is separately checked for exact column order, duplicate authors, duplicate indices, missing interior indices, the exact index set, integer dtype, string author storage, and numeric author ordering.

**Stop conditions:** stop on any supplemental invariant or published checksum mismatch.

In [ ]:
import pandas as pd

from tdmec.hashing import sha256_file

PUBLISHED_FRAME = pd.read_parquet(LOCAL_CANONICAL_NODE_MAP)
PUBLISHED_VALIDATION = validate_canonical_node_map_frame(
    PUBLISHED_FRAME,
    expected_count=16_736,
)
assert PUBLISHED_FRAME.columns.tolist() == [
    "author_account_id",
    "node_index",
]
assert PUBLISHED_FRAME["author_account_id"].nunique() == 16_736
assert PUBLISHED_FRAME["node_index"].nunique() == 16_736
assert set(PUBLISHED_FRAME["node_index"].astype(int)) == set(range(16_736))
assert PUBLISHED_VALIDATION.exact_index_set is True
assert PUBLISHED_VALIDATION.canonical_numeric_order is True
assert sha256_file(LOCAL_CANONICAL_NODE_MAP) == PUBLISHED_NODE_MAP.sha256
print("Published row count:", PUBLISHED_VALIDATION.row_count)
print("Published unique author count:", PUBLISHED_VALIDATION.unique_author_count)
print("Published unique index count:", PUBLISHED_VALIDATION.unique_index_count)
print("PASS: Supplemental published-artifact validation succeeded.")

## Stage 15 - Write and persist the privacy-safe validation manifest

The manifest contains only the approved aggregate fields, filename, commit, timestamp, and SHA-256 of the accepted Drive canonical file. It is atomically written locally, then created or revision-updated through the Drive API. It contains no author IDs, usernames, tweet text, email addresses, credentials, Drive file IDs, or private absolute paths.

**Stop conditions:** stop on an unexpected field, privacy failure, invalid aggregate, or local/Drive manifest-write failure.

In [ ]:
from datetime import datetime, timezone

from tdmec_diagnostics.privacy import assert_privacy_safe_mapping

EXECUTION_TIMESTAMP = datetime.now(timezone.utc).isoformat(
    timespec="seconds"
).replace("+00:00", "Z")
VALIDATION_RECORD = build_validation_manifest(
    published=PUBLISHED_NODE_MAP,
    scan=DATASET_A_SCAN,
    audited_repository_commit=EXPECTED_SHA,
    execution_timestamp=EXECUTION_TIMESTAMP,
)
APPROVED_MANIFEST_FIELDS = {
    "artifact_type",
    "canonical_filename",
    "sha256",
    "row_count",
    "columns",
    "index_min",
    "index_max",
    "unique_author_count",
    "unique_index_count",
    "exact_index_set",
    "canonical_numeric_order",
    "dataset_a_workbook_count",
    "total_rows_inspected",
    "valid_author_record_count",
    "missing_author_record_count",
    "malformed_author_record_count",
    "audited_repository_commit",
    "execution_timestamp",
}
assert set(VALIDATION_RECORD) == APPROVED_MANIFEST_FIELDS
assert_privacy_safe_mapping(VALIDATION_RECORD)
write_validation_manifest_atomic(
    VALIDATION_RECORD,
    LOCAL_VALIDATION_MANIFEST,
)
assert LOCAL_VALIDATION_MANIFEST.is_file()
REMOTE_VALIDATION_MANIFEST = drive_replace_json(
    MANIFESTS_FOLDER_ID,
    LOCAL_VALIDATION_MANIFEST.name,
    VALIDATION_RECORD,
)
print("Validation-manifest filename:", LOCAL_VALIDATION_MANIFEST.name)
print("PASS: Privacy-safe validation manifest persisted to Drive.")

## Stage 16 - Final fail-closed status

This final stage downloads the persistent Drive canonical map and manifest again, then rechecks their checksum, content, and aggregate agreement. A PASS authorizes the canonical node map as the future node universe; it does not run Phase 2 diagnostics, build graph features, execute later phases, or implement Phase 3.

**Stop conditions:** any final checksum, manifest, count, or malformed-record disagreement produces a hard stop.

In [ ]:
import json

FINAL_MANIFEST = json.loads(
    drive_download_bytes(REMOTE_VALIDATION_MANIFEST["id"])
)
assert FINAL_MANIFEST == VALIDATION_RECORD
FINAL_CANONICAL_CHECK = LOCAL_OUTPUT_ROOT / ".final-node-map-check.parquet"
FINAL_CANONICAL_CHECK.unlink(missing_ok=True)
drive_download_file(REMOTE_CANONICAL_FILE, FINAL_CANONICAL_CHECK)
assert FINAL_MANIFEST["sha256"] == sha256_file(FINAL_CANONICAL_CHECK)
FINAL_FRAME = pd.read_parquet(FINAL_CANONICAL_CHECK)
validate_canonical_node_map_frame(FINAL_FRAME, expected_count=16_736)
assert FINAL_FRAME.reset_index(drop=True).equals(
    CANONICAL_FRAME.reset_index(drop=True)
)
FINAL_CANONICAL_CHECK.unlink(missing_ok=True)
assert FINAL_MANIFEST["row_count"] == 16_736
assert FINAL_MANIFEST["index_min"] == 0
assert FINAL_MANIFEST["index_max"] == 16_735
assert FINAL_MANIFEST["unique_author_count"] == 16_736
assert FINAL_MANIFEST["unique_index_count"] == 16_736
assert FINAL_MANIFEST["exact_index_set"] is True
assert FINAL_MANIFEST["canonical_numeric_order"] is True
assert FINAL_MANIFEST["dataset_a_workbook_count"] == 12
assert FINAL_MANIFEST["malformed_author_record_count"] == 0

print("Canonical filename:", CANONICAL_FILENAME)
print("Canonical SHA-256:", FINAL_MANIFEST["sha256"])
print("Unique author count:", FINAL_MANIFEST["unique_author_count"])
print("FINAL STATUS: PASS - DATASET A CANONICAL NODE MAP VALIDATED")